In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
df_full = pd.read_parquet("../data/raw/civil_comments/train/")
print(df_full.shape)

## Random sample (200,000 comments)

In [ ]:
df_sample = df_full.sample(200_000, random_state=42)
df_sample.to_parquet("../data/processed/civil_comments_sample.parquet")
print(df_sample.shape)

In [ ]:
df = pd.read_parquet("../data/raw/civil_comments/")
print(df.shape)

In [ ]:
labels = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit",
]

# A label counts as present if at least half of the annotators agreed
count = (df_sample[labels] >= 0.5).sum()
share = (count / len(df_sample) * 100).round(2)

pd.DataFrame({"count": count, "share_percent": share})

In [ ]:
# Compare with the full dataset
full_count = (df_full[labels] >= 0.5).sum()
pd.DataFrame({"sample": count, "full": full_count})

## Building the training set
Rare categories are underrepresented in the sample. All comments from the full
dataset where obscene, threat, identity_attack or sexual_explicit >= 0.5 are added.
severe_toxicity is dropped (only 13 cases overall); severity is derived from the
toxicity score instead. Evaluation uses the official validation/test splits.

In [ ]:
rare_labels = ["obscene", "threat", "identity_attack", "sexual_explicit"]
is_rare = (df_full[rare_labels] >= 0.5).any(axis=1)

# Combine sample and rare cases, remove duplicate texts
df_train = pd.concat([df_sample, df_full[is_rare]]).drop_duplicates(subset="text")

print(df_train.shape)
(df_train[labels] >= 0.5).mean().mul(100).round(2)

In [ ]:
# Free memory: the full dataset is no longer needed
del df_full

## Validation and test data

In [ ]:
df_val = pd.read_parquet("../data/raw/civil_comments/validation/")
df_test = pd.read_parquet("../data/raw/civil_comments/test/")

print(df_val.shape, df_test.shape)
(df_val[labels] >= 0.5).mean().mul(100).round(2)

## Data leakage check
Do identical comment texts appear in both training and evaluation data?

In [ ]:
overlap_val = df_train["text"].isin(df_val["text"]).sum()
overlap_test = df_train["text"].isin(df_test["text"]).sum()

print(f"Training texts also in validation: {overlap_val}")
print(f"Training texts also in test: {overlap_test}")

In [ ]:
# True if the training text also appears in validation or test
in_eval = df_train["text"].isin(df_val["text"]) | df_train["text"].isin(df_test["text"])

print(f"Overlapping texts: {in_eval.sum()}")
print(df_train.loc[in_eval, "text"].str.len().describe())
df_train.loc[in_eval, "text"].head(10)

In [ ]:
# Keep only texts that do not appear in the evaluation data
df_train = df_train[~in_eval]
print(df_train.shape)

df_train.to_parquet("../data/processed/civil_comments_train_enriched.parquet")

# Check: both overlaps should now be 0
print(df_train["text"].isin(df_val["text"]).sum(), df_train["text"].isin(df_test["text"]).sum())

## Comment length

In [ ]:
df_train["length"] = df_train["text"].str.len()
df_train["is_toxic"] = df_train["toxicity"] >= 0.5

print(df_train["length"].describe())
print(f"\nComments with exactly 1000 characters: {(df_train['length'] == 1000).sum()}")
print(f"Comments with more than 1000 characters: {(df_train['length'] > 1000).sum()}")

df_train.groupby("is_toxic")["length"].median()

In [ ]:
df_train["length"].plot(kind="hist", bins=50, title="Comment length (characters)")
plt.xlabel("Characters")
plt.show()

**Findings:** Right-skewed length distribution (median 201 characters).
Spike near 1,000 characters suggests a platform character limit (only 2 comments exceed it).
Toxic comments are slightly shorter (median 184 vs. 205), so length alone is not a useful signal.
A maximum of 256 tokens should cover almost all comments for mmBERT.

## Toxicity score distribution

In [ ]:
print(df_train["toxicity"].describe())

df_train["toxicity"].plot(kind="hist", bins=50, title="Toxicity score")
plt.xlabel("Share of annotators rating the comment as toxic")
plt.show()

## Correlation between labels

In [ ]:
sns.heatmap(df_train[labels].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation between labels")
plt.show()

## Most distinctive words in toxic comments

In [ ]:
# Count words, ignore common English filler words and very rare words
vectorizer = CountVectorizer(stop_words="english", min_df=20)
counts = vectorizer.fit_transform(df_train["text"])
words = vectorizer.get_feature_names_out()

is_toxic = df_train["is_toxic"].values
toxic_counts = counts[is_toxic].sum(axis=0).A1
clean_counts = counts[~is_toxic].sum(axis=0).A1

df_words = pd.DataFrame({"word": words, "toxic": toxic_counts, "clean": clean_counts})
df_words["toxic_share"] = df_words["toxic"] / (df_words["toxic"] + df_words["clean"])

# Words that appear often enough and mostly in toxic comments
df_words[df_words["toxic"] >= 200].sort_values("toxic_share", ascending=False).head(20)

In [ ]:
# Experiment: the same analysis without a minimum document frequency
vectorizer_all = CountVectorizer(stop_words="english", min_df=1)
counts_all = vectorizer_all.fit_transform(df_train["text"])
words_all = vectorizer_all.get_feature_names_out()

toxic_all = counts_all[is_toxic].sum(axis=0).A1
clean_all = counts_all[~is_toxic].sum(axis=0).A1
df_all = pd.DataFrame({"word": words_all, "toxic": toxic_all, "clean": clean_all})
df_all["toxic_share"] = df_all["toxic"] / (df_all["toxic"] + df_all["clean"])

print(f"Vocabulary with min_df=1:  {len(words_all)}")
print(f"Vocabulary with min_df=20: {len(words)}")
print(f"Words with toxic_share = 100 %: {(df_all['toxic_share'] == 1).sum()}")

del vectorizer_all, counts_all, df_all

**Finding:** Without min_df, the vocabulary grows from 17,950 to 112,327 words,
and 8,586 words get a toxic share of 100 % purely by chance (mostly typos and noise).
min_df=20 removes this noise. Vulgar is not the same as toxic ("damn", "sucks").
Obfuscated spellings (e.g. "a$$") are lost by word-level tokenization → character n-grams in Phase 3.

In [ ]:
# How word-level and character-level tokenization treat obfuscated spellings
analyzer = CountVectorizer().build_analyzer()
char_analyzer = CountVectorizer(analyzer="char_wb", ngram_range=(2, 4)).build_analyzer()

print(analyzer("You are an A$$ and an id10t"))
print(char_analyzer("A$$"))

In [ ]:
# Same analysis without a minimum: every word is kept
vectorizer_all = CountVectorizer(stop_words="english", min_df=1)
counts_all = vectorizer_all.fit_transform(df_train["text"])
words_all = vectorizer_all.get_feature_names_out()

toxic_all = counts_all[is_toxic].sum(axis=0).A1
clean_all = counts_all[~is_toxic].sum(axis=0).A1

df_all = pd.DataFrame({"word": words_all, "toxic": toxic_all, "clean": clean_all})
df_all["toxic_share"] = df_all["toxic"] / (df_all["toxic"] + df_all["clean"])

print(f"Vocabulary with min_df=1:  {len(words_all)}")
print(f"Vocabulary with min_df=20: {len(words)}")
print(f"Words with toxic_share = 100 %: {(df_all['toxic_share'] == 1).sum()}")

df_all.sort_values(["toxic_share", "toxic"], ascending=[False, True]).head(10)

## Identity terms (unintended bias)

In [ ]:
identity_words = ["gay", "muslim", "black", "white", "jewish", "christian", "women", "trans"]
df_words[df_words["word"].isin(identity_words)].sort_values("toxic_share", ascending=False)

In [ ]:
# Same word counts on the unmodified validation data
counts_val = vectorizer.transform(df_val["text"])
is_toxic_val = (df_val["toxicity"] >= 0.5).values

df_words_val = pd.DataFrame({
    "word": words,
    "toxic": counts_val[is_toxic_val].sum(axis=0).A1,
    "clean": counts_val[~is_toxic_val].sum(axis=0).A1,
})
df_words_val["toxic_share"] = df_words_val["toxic"] / (df_words_val["toxic"] + df_words_val["clean"])

df_words_val[df_words_val["word"].isin(identity_words)].sort_values("toxic_share", ascending=False)

**Findings (identity terms):** On the unmodified validation data (base rate ~8 %),
comments containing "black", "gay", "white" or "muslim" are toxic about 3x as often.
These words are not offensive themselves, but models may learn them as shortcuts
(unintended bias). The enriched training set inflates these shares, so statistics
must be checked on unmodified data. Small counts (e.g. "trans": 130) are unreliable.
To be tested in Phase 7 with neutral template sentences per identity group.

## Summary of findings (English data)
- **Data quality:** No missing values or empty texts. 658 training texts overlapping
  with validation/test were removed (data leakage).
- **Labels:** 8 % toxic in the natural distribution. Rare categories (threat 0.26 %)
  were enriched from the full dataset. severe_toxicity dropped (13 cases overall).
- **Toxicity score:** Strongly right-skewed, median 0. Many comments between 0 and 0.5
  where only some annotators saw toxicity → soft labels in Phase 5.
- **Length:** Median 201 characters, platform limit at 1,000. Length alone is not a useful
  signal. Max. 256 tokens should cover almost all comments.
- **Words:** Vulgar is not the same as toxic ("damn", "sucks"). Obfuscated spellings
  (e.g. "a$$") are lost by word-level tokenization → character n-grams in Phase 3.
- **Bias:** Identity terms ("black", "gay", "muslim") appear about 3x as often in toxic
  comments → to be tested in Phase 7.
- **Label correlation:** insult ≈ toxicity (0.87); threat (0.37) and sexual_explicit (0.35)
  are distinct signals → multi-label model in Phase 5.